In [ ]:
# # Blender + Flux Depth-Inpaint smoke (Kaggle P100, NF4)
#
# Повний pipeline на Kaggle БЕЗ RunPod: Stage 1 (BlenderProc render) → Stage 3 (Flux Depth-dev NF4 inpaint) → Stage 4 (composite + bbox).
#
# Мета: швидко iterate prompt template на real Flux output БЕЗ оплати RunPod. NF4 quant = ~93% якості bf16, fits 16GB P100 з cpu_offload.
#
# ## Передумови
# 1. Kaggle dataset `dariachuprina/yolo-bluebird-df-assets` attached як Input.
# 2. Accelerator: **GPU P100** (Settings → Accelerator).
# 3. Internet: **On**.
# 4. **HF_TOKEN** — один з трьох шляхів (див. Cell 1).
#
# Repo тимчасово public — GITHUB_TOKEN не потрібен. Після smoke privatize назад.

In [ ]:
# Cell 1 — env + HF_TOKEN
#
# HF_TOKEN — три шляхи (один з них):
#   (A) Kaggle Secrets: Add-ons → Secrets → Add HF_TOKEN → Attach to notebook
#   (B) Kaggle Variables: right sidebar → Variables → HF_TOKEN = hf_xxx
#   (C) Inline paste нижче (закоментоване), run, але НЕ Save Version
#
import os, sys, torch

# (C) inline fallback — раскоментуй ОДНУ строку, встав свій token, run.
# os.environ['HF_TOKEN'] = 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# (A) Kaggle Secrets automatic loading
if 'HF_TOKEN' not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(
            f'HF_TOKEN не set. Опції: '
            f'(A) Add-ons→Secrets→Attach HF_TOKEN; '
            f'(B) Variables sidebar; '
            f'(C) uncomment+paste у Cell 1. '
            f'Kaggle secret error: {e}'
        )

print(f'Python: {sys.version.split()[0]}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
assert torch.cuda.is_available(), 'enable GPU P100 у Settings → Accelerator'
print(f"HF_TOKEN: {os.environ['HF_TOKEN'][:8]}...")
print('(GITHUB_TOKEN не потрібен — repo public)')

In [ ]:
# Cell 2 — install diffusion stack + render stack
!pip install -q diffusers>=0.31 transformers>=4.44 accelerate>=0.33 sentencepiece safetensors bitsandbytes>=0.43
!pip install -q blenderproc h5py pyyaml opencv-python-headless
print('✓ deps installed')

In [ ]:
# Cell 3 — clone repo (public) + auto-detect assets
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working/yolo-bluebierd')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)  # очистити stale partial clone

subprocess.run(['git', 'clone',
                'https://github.com/ChuprinaDaria/YOLO-Bluebierd.git',
                str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

INPUT = Path('/kaggle/input')
matches = list(INPUT.rglob('hdri'))
assert matches, 'attach dataset yolo-bluebird-df-assets як Input'
ASSETS = matches[0].parent
print(f'repo: {REPO_DIR}')
print(f'assets: {ASSETS}')
for sub in ('hdri', 'textures/ground', 'models/light_vehicle'):
    n = len(list((ASSETS / sub).rglob('*')))
    print(f'  {sub}: {n} items')


In [ ]:
# Cell 4 — BlenderProc quickstart (downloads Blender ~330 MB на /root/blender)
!rm -rf /root/blender 2>/dev/null
!blenderproc quickstart 2>&1 | tail -5
print('\n✓ Blender ready')

In [ ]:
# Cell 5 — Stage 1: render 1 frame з diffusion preset
import os, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/kaggle/working/yolo-bluebierd')
ASSETS = Path([p for p in Path('/kaggle/input').rglob('hdri')][0].parent)
CFG_SRC = REPO_DIR / 'datasetforge/configs/v1_light_vehicle.yaml'
OUT = Path('/kaggle/working/v2_smoke_1frame')
OUT.mkdir(parents=True, exist_ok=True)

# Override config: use diffusion image size (1024x1024)
cfg = yaml.safe_load(CFG_SRC.read_text())
inf_size = cfg.get('image_size_diffusion', [1024, 1024])
cfg['image_size'] = inf_size
cfg['diffusion']['enabled'] = True
CFG_RUN = OUT / 'v1_light_vehicle_diff.yaml'
CFG_RUN.write_text(yaml.safe_dump(cfg))
print(f'render @ {inf_size}, cfg → {CFG_RUN}')
print(f'prompt template:\n  {cfg["diffusion"]["prompt_template"]}\n')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
cmd = ['blenderproc', 'run', str(REPO_DIR / 'datasetforge/engine/render_runner.py'),
       '--config', str(CFG_RUN), '--n', '1', '--out', str(OUT),
       '--assets-root', str(ASSETS), '--seed', '42']
print('[run]', ' '.join(cmd))
subprocess.run(cmd, env=env, check=True)

print('\n✓ Stage 1 done')
for sub in ['images','labels','depth','normals','vehicle_masks','metadata']:
    files = list((OUT / sub).glob('*'))
    print(f'  {sub}: {len(files)} file(s)')

In [ ]:
# Cell 6 — Stage 1 preview (RGB+bbox / depth / normals / mask) + metadata
import json, cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

OUT = Path('/kaggle/working/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT / 'images').glob('*.jpg'))[0]

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
depth = cv2.imread(str(OUT/'depth'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
normals_raw = cv2.imread(str(OUT/'normals'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
meta = json.loads((OUT/'metadata'/f'{stem}.json').read_text())

H, W = rgb.shape[:2]
rgb_pil = Image.fromarray(rgb).copy()
d = ImageDraw.Draw(rgb_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=4)

normals_viz = ((normals_raw.astype(np.float32)/65535.0)*255).astype(np.uint8) if normals_raw.dtype==np.uint16 else normals_raw

fig, ax = plt.subplots(2, 2, figsize=(14, 14))
ax[0,0].imshow(rgb_pil); ax[0,0].set_title('Stage 1 RGB + bbox'); ax[0,0].axis('off')
ax[0,1].imshow(depth, cmap='viridis'); ax[0,1].set_title(f'Depth max={int(depth.max())} mm'); ax[0,1].axis('off')
ax[1,0].imshow(normals_viz); ax[1,0].set_title('Normals (encoded)'); ax[1,0].axis('off')
ax[1,1].imshow(mask, cmap='gray'); ax[1,1].set_title(f'Mask veh_px={int((mask>=128).sum())}'); ax[1,1].axis('off')
plt.tight_layout(); plt.show()

print(f"\nFrame metadata:")
print(f"  sun: {meta['sun_cardinal']} @ {meta['sun_elevation_deg']:.0f}° elev")
print(f"  season={meta['season']} weather={meta['weather']} landscape={meta['landscape']}")
print(f"  distance={meta['distance_m']}m altitude={meta['altitude_m']:.0f}m angle={meta['view_angle_deg']}°")
print(f"  model={meta['model_variant']}")

In [ ]:
# Cell 7 — Load Flux Depth-dev NF4 + main pipeline. ~5-8 min на первому запуску
# (download ~10 GB models, потім кеш у /root/.cache/huggingface)
import time, torch
from diffusers import FluxControlInpaintPipeline, FluxTransformer2DModel
from transformers import T5EncoderModel

t0 = time.time()
print('[load] FLUX.1-Depth-dev-nf4 transformer (NF4 quant ~6GB)...')
transformer = FluxTransformer2DModel.from_pretrained(
    'diffusers/FLUX.1-Depth-dev-nf4',
    subfolder='transformer',
    torch_dtype=torch.bfloat16,
)
print(f'  {time.time()-t0:.0f}s')

t1 = time.time()
print('[load] T5 encoder NF4 (~3 GB)...')
text_encoder_2 = T5EncoderModel.from_pretrained(
    'diffusers/FLUX.1-Depth-dev-nf4',
    subfolder='text_encoder_2',
    torch_dtype=torch.bfloat16,
)
print(f'  {time.time()-t1:.0f}s')

t2 = time.time()
print('[load] main pipeline (CLIP + VAE + scheduler, ~1 GB)...')
pipe = FluxControlInpaintPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-Depth-dev',
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    torch_dtype=torch.bfloat16,
)
print(f'  {time.time()-t2:.0f}s')

pipe.enable_model_cpu_offload()
torch.cuda.synchronize()
print(f'\n✓ pipeline ready in {time.time()-t0:.0f}s total')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f}/{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 8 — Stage 3: depth-conditioned inpaint. ~60-120s на P100
import time, yaml
from pathlib import Path
from datasetforge.pipelines.inpaint import bg_filler

OUT = Path('/kaggle/working/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
cfg = yaml.safe_load((OUT/'v1_light_vehicle_diff.yaml').read_text())
diff_cfg = cfg['diffusion']
(OUT/'ai_bg').mkdir(exist_ok=True)

t0 = time.time()
sidecar = bg_filler.inpaint_one(
    pipe,
    rgb_path=OUT/'images'/f'{stem}.jpg',
    depth_path=OUT/'depth'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'ai_bg'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
)
print(f'\n✓ inpaint {time.time()-t0:.0f}s | seed={sidecar["seed"]}')
print(f'\nprompt used:\n  {sidecar["prompt"]}')
print(f'\nnegative:\n  {sidecar["negative_prompt"] or "<not supported by pipeline>"}')

In [ ]:
# Cell 9 — Stage 4: composite + 3-up preview + pixel-identity check
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
from datasetforge.pipelines.inpaint import composite

OUT = Path('/kaggle/working/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
(OUT/'composite').mkdir(exist_ok=True)

stats = composite.composite_one(
    rgb_path=OUT/'images'/f'{stem}.jpg',
    ai_bg_path=OUT/'ai_bg'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    normals_path=OUT/'normals'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'composite'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
    assert_pixel_identity=True,
)
print(f'✓ composite OK: {stats}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
ai_bg = np.array(Image.open(OUT/'ai_bg'/f'{stem}.png'))
comp = np.array(Image.open(OUT/'composite'/f'{stem}.png'))
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
mb = mask >= 128
diff = int(np.abs(rgb.astype(int) - comp.astype(int))[mb].max())
print(f'\nvehicle pixel max-diff: {diff} (relight OFF → expect 0)')

H, W = comp.shape[:2]
comp_pil = Image.fromarray(comp).copy()
d = ImageDraw.Draw(comp_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=4)

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ax[0].imshow(rgb); ax[0].set_title('Stage 1 — raw 3D render'); ax[0].axis('off')
ax[1].imshow(ai_bg); ax[1].set_title('Stage 3 — AI background (Flux NF4)'); ax[1].axis('off')
ax[2].imshow(comp_pil); ax[2].set_title(f'Stage 4 — composite + bbox (vehicle drift={diff})'); ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ## ⚖️ Review
#
# Подивись на composite:
# 1. **Vehicle pixels intact?** `vehicle drift = 0` має бути. Якщо НЕ 0 — баг у composite (репорт у чат).
# 2. **Bbox tight?** Зелена рамка має точно охоплювати техніку.
# 3. **AI background:** виглядає як реальне аерофото для season/landscape/sun?
# 4. **Тіні:** напрям тіней на vehicle і AI bg збігається?
# 5. **AI tells:** немає melted text / impossible geometry / tiling?
#
# **Якщо погано** — змінити `prompt_template` у `datasetforge/configs/v1_light_vehicle.yaml`, commit + push, у Cell 3 зробити `git pull`, re-run cells 5-9. Iterate.
#
# **Якщо добре** — переходимо на RunPod з full bf16 (вищий quality).
#
# Раз notebook прогнаний — Save як Version (Kaggle збереже output) щоб не втратити.

In [ ]:
# Cell 11 (опц) — try другий seed з тим же prompt — посмотреть variance
import time, json
from pathlib import Path

OUT = Path('/kaggle/working/v2_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
meta_path = OUT/'metadata'/f'{stem}.json'
meta = json.loads(meta_path.read_text())

# Sweep seeds
alt_dir = OUT/'ai_bg_seeds'
alt_dir.mkdir(exist_ok=True)
for seed_inc in (1, 2, 3):
    meta['seed'] = 42 + 1000*seed_inc
    meta_alt = meta_path.parent / f'{stem}_seed{seed_inc}.json'
    meta_alt.write_text(json.dumps(meta))
    t0 = time.time()
    bg_filler.inpaint_one(pipe,
        rgb_path=OUT/'images'/f'{stem}.jpg', depth_path=OUT/'depth'/f'{stem}.png',
        mask_path=OUT/'vehicle_masks'/f'{stem}.png', meta_path=meta_alt,
        out_path=alt_dir/f'{stem}_seed{seed_inc}.png', diffusion_cfg=diff_cfg)
    print(f'seed {seed_inc}: {time.time()-t0:.0f}s')

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
fig, ax = plt.subplots(1, 3, figsize=(21, 7))
for i, s in enumerate((1, 2, 3)):
    ax[i].imshow(np.array(Image.open(alt_dir/f'{stem}_seed{s}.png'))); 
    ax[i].set_title(f'seed +{s}'); ax[i].axis('off')
plt.tight_layout(); plt.show()